In [ ]:
import os, sys
sys.path.append('../')
import MeshFEM, mesh, benchmark
import argparse
import numpy as np
import time
import plot_video_utils
import matplotlib
from matplotlib import pyplot as plt
from matplotlib.ticker import MaxNLocator
from matplotlib.legend_handler import HandlerTuple
import MeshFEMParamSolverEnum as SolverOptionEnum

In [ ]:
import math

# User Input

In [ ]:
base_path = 'EnumSettingExps/GridModelSet2_exps_0819/'

In [ ]:
thread_num_list = [8]

In [ ]:
modelbase_path = '../../../Models/GridModelSet2/'

In [ ]:
# Check if the modelbase path exists
if not os.path.exists(modelbase_path):
    print(f"Error: The specified model base path '{modelbase_path}' does not exist.")
    sys.exit(1)

# List all files in the directory and filter for 3D model files
model_files = [f for f in os.listdir(modelbase_path) if f.endswith(('.obj', '.msh', '.off'))]

if not model_files: raise RuntimeWarning(f"No obj, msh, off files in {modelbase_path}")

In [ ]:
# delete 'TeddyBear_1605K_fig10_4.obj'
# model_files.remove('TeddyBear_1605K_fig10_4.obj')

In [ ]:
from pathlib import Path
model_names = [Path(f).stem for f in model_files]
print(model_names)

In [ ]:
# model_names = ['bunny_cut'] 

In [ ]:
num_models = len(model_names)
print("Number of Models in List: ", num_models)

# Pair List and Other Settings

In [ ]:
settingName = 'ProjectionType'
filteredSettingName = 'EigenvalueModification'
keep_option_index = 0 # Keep 'Clamp'
option_name_tuple = SolverOptionEnum.get_setting_options(settingName)

In [ ]:
ori_pairs_tuple_list = SolverOptionEnum.variant_pairs_by_setting(settingName)
filtered_pairs_tuple_list = SolverOptionEnum.filter_pairs_by_option(ori_pairs_tuple_list, filteredSettingName, keep_option_index)

In [ ]:
filtered_pairs_tuple_list

In [ ]:
numPairs = len(filtered_pairs_tuple_list)
color_list, line_style_list, line_width_list = plot_video_utils.getColorLineList(numPairs)
sect = None
Metric_name = "Grad Norm"
Metric_ind = 1

In [ ]:
line_style_list = ['-', ':'] # First Solid, Second dotted

# Grid Plots

In [ ]:
numCols = 3
numRows = math.ceil(num_models/numCols)
print(numRows)

In [ ]:
single_fig_width = 6
single_fig_height = 6
interval = 2
grid_fig_wdith = numCols * single_fig_width + (numCols - 1) * interval
grid_fig_height = numRows * single_fig_height + (numRows - 1) * interval

In [ ]:
grid_fig_wdith

## Grad Norm VS Iteration

In [ ]:
# Test add hessianWasProjected and add HessianIndefinite
addScatter = False
scatterType = 'indefinite' # 'indefinite', 'all'

In [ ]:
def head_slicing(a, n):
    a = np.asarray(a)
    return a if n is None else a[:n]

In [ ]:
plt.figure(figsize=(grid_fig_wdith, grid_fig_height))

for model_ind, model_name in enumerate(model_names):
    model_stats_dir = os.path.join(base_path, model_name)
    
    plt.subplot(numRows, numCols, model_ind+1)
    handles_tuple_list = []
    labels_tuple_list = []
    
    for pair_indx in range(numPairs):
        meshfemsolver_option_list = [f'MeshFEM{i}' for i in filtered_pairs_tuple_list[pair_indx]]
        # read data in result folder
        obj_grad_time_list, benchmarkdict_list, hessian_stats_list = plot_video_utils.readConvergenceTimingData(model_stats_dir, thread_num_list, meshfemsolver_option_list, True)
        
        thread_ind = 0 # only 8-thread exp

        num_options = len(meshfemsolver_option_list)
        iterations_list = []
        for i in range(num_options):
            iterations = np.arange(0, obj_grad_time_list[i][thread_ind].shape[1])
            iterations_list.append(iterations)
        
        # Line plot
        pair_line_handlers = []
        for i in range(num_options):
            x_data_iterations = head_slicing(iterations_list[i], sect)
            y_data_grad_norms = head_slicing(obj_grad_time_list[i][thread_ind][Metric_ind], sect)
            
            lhand, = plt.plot(x_data_iterations, y_data_grad_norms, 
                             ls=line_style_list[i], lw=line_width_list[pair_indx], color=color_list[pair_indx])
            pair_line_handlers.append(lhand)
            
            # add Scatter
            if addScatter:
                if scatterType == 'projected':
                    hessian_projected_array = hessian_stats_list[i][thread_ind].projected
                    proj = (hessian_projected_array == 1) # Boolean Mask
                    proj_sliced = head_slicing(proj, sect)
                    plt.scatter(x_data_iterations[proj_sliced], y_data_grad_norms[proj_sliced],
                               marker='o', edgecolors=color_list[pair_indx], facecolors=color_list[pair_indx], s=50)
                elif scatterType == 'indefinite':
                    hessian_indefinite_array = hessian_stats_list[i][thread_ind].indefinite
                    indef = (hessian_indefinite_array == 1) # Boolean Mask
                    indef_sliced = head_slicing(indef, sect)
                    plt.scatter(x_data_iterations[indef_sliced], y_data_grad_norms[indef_sliced],
                               marker='D', edgecolors=color_list[pair_indx], facecolors='none', s=60)

        handles_tuple_list.append(tuple(pair_line_handlers))
        labels_tuple_list.append(SolverOptionEnum.get_pairline_label(filtered_pairs_tuple_list, settingName, pair_indx))
        
    
    # Axis and Legend of plt
    plt.title(f"Model: {model_name} - \"{settingName}\": {option_name_tuple[0]} vs {option_name_tuple[1]}", fontsize=10)
    plt.yscale('log')
    plt.xlabel("Iteration", fontsize=10)
    plt.ylabel(Metric_name, fontsize=10)
    plt.legend(handles_tuple_list, labels_tuple_list,
                   handler_map={tuple: HandlerTuple(ndivide=None)}, fontsize=8)

    # Ensure x-axis values are only positive integers
    ax = plt.gca()  # Get the current axis
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))  # Force integer x-axis ticks
    ax.set_xlim(left=0)
    ax.set_ylim(bottom=0)

plt.show()

## Grad Norm VS time

In [ ]:
plt.figure(figsize=(grid_fig_wdith, grid_fig_height))

for model_ind, model_name in enumerate(model_names):
    model_stats_dir = os.path.join(base_path, model_name)
    
    plt.subplot(numRows, numCols, model_ind+1)
    handles_tuple_list = []
    labels_tuple_list = []
    
    for pair_indx in range(numPairs):
        meshfemsolver_option_list = [f'MeshFEM{i}' for i in filtered_pairs_tuple_list[pair_indx]]
        # read data in result folder
        obj_grad_time_list, benchmarkdict_list = plot_video_utils.readConvergenceTimingData(model_stats_dir, thread_num_list, meshfemsolver_option_list)
        
        thread_ind = 0 # only 8-thread exp

        num_options = len(meshfemsolver_option_list)
        iterations_list = []
        for i in range(num_options):
            iterations = np.arange(0, obj_grad_time_list[i][thread_ind].shape[1])
            iterations_list.append(iterations)

        pair_line_handlers = []
        for i in range(num_options):
            if sect is not None:  lhand, = plt.plot(obj_grad_time_list[i][thread_ind][2][:sect], obj_grad_time_list[i][thread_ind][Metric_ind][:sect], ls=line_style_list[i], lw=line_width_list[pair_indx],color=color_list[pair_indx])
            else:                 lhand, = plt.plot(obj_grad_time_list[i][thread_ind][2], obj_grad_time_list[i][thread_ind][Metric_ind], ls=line_style_list[i], lw=line_width_list[pair_indx], color=color_list[pair_indx])
            pair_line_handlers.append(lhand)

        handles_tuple_list.append(tuple(pair_line_handlers))
        labels_tuple_list.append(SolverOptionEnum.get_pairline_label(filtered_pairs_tuple_list, settingName, pair_indx))
    
    # Axis and Legend of plt
    plt.title(f"Model: {model_name} - \"{settingName}\": {option_name_tuple[0]} vs {option_name_tuple[1]}", fontsize=10)
    plt.yscale('log')
    plt.xlabel("Time [sec]", fontsize=10)
    plt.ylabel(Metric_name, fontsize=10)
    plt.legend(handles_tuple_list, labels_tuple_list,
                   handler_map={tuple: HandlerTuple(ndivide=None)}, fontsize=8)

    # Ensure x-axis values are only positive integers
    ax = plt.gca()  # Get the current axis
    ax.set_xlim(left=0)
    ax.set_ylim(bottom=0)

plt.show()